# 🏎️ F1 Race Prediction — Gerçek Veriyle Model Eğitimi (Google Colab)

Bu notebook, `models/ml_predictor.py` içindeki sentetik eğitim verisinin yerine **gerçek F1 sonuçlarıyla** (FastF1) bir model eğitir.

**Kullanım**
1. `Runtime → Change runtime type → T4 GPU` (XGBoost GPU'yu kullanır; RF/GB zaten CPU'da çalışır — veri küçük olduğu için CPU runtime da yeterli)
2. `Runtime → Run all`
3. Sonunda `f1_model.joblib` ve `f1_training_data.csv` otomatik indirilir → projede `models/trained/` klasörüne koy.

⏱️ Veri indirme birkaç dakika sürer. Cache'i kalıcı yapmak için aşağıda `USE_DRIVE = True` yap; sonraki çalıştırmalar dakikalar sürer.

**Değerlendirme:** Zamana göre ayrılır → eğitim ≤ 2024, doğrulama 2025, test 2026 (sezonun tamamlanan yarışları). Rastgele ayırma geleceği sızdırır, bu yüzden kullanılmıyor.

In [ ]:
!pip install -q fastf1 xgboost joblib
!nvidia-smi -L || echo "GPU yok - CPU ile devam"

In [ ]:
import os, time, warnings, logging, subprocess
import numpy as np
import pandas as pd
import fastf1

warnings.filterwarnings('ignore')
fastf1.set_log_level('ERROR')
logging.disable(logging.WARNING)   # FastF1'in INFO/WARNING kalabalığını gizle

YEARS = list(range(2018, 2027))   # 2026'nın tamamlanmış yarışları da dahil
USE_DRIVE = False                 # True → cache Google Drive'da kalır, tekrar indirme olmaz

CACHE_DIR = '/content/fastf1_cache'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/f1_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
fastf1.Cache.enable_cache(CACHE_DIR)

HAS_GPU = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
DEVICE = 'cuda' if HAS_GPU else 'cpu'
print('XGBoost device:', DEVICE)

## 1. Veri indirme (Jolpica/Ergast: yarış + sıralama sonuçları, FastF1: hava durumu)

In [ ]:
import requests

try:
    from fastf1 import _api as f1api      # FastF1 >= 3.5
except ImportError:
    from fastf1 import api as f1api

API = 'https://api.jolpi.ca/ergast/f1'   # Ergast'ın devamı; sezonu tek seferde, sayfa sayfa verir


def jolpica(path):
    # Bir sezonun tüm sonuçlarını sayfalayarak çek → {round: {...race, 'rows': [...]}}
    races, offset = {}, 0
    while True:
        for attempt in range(8):
            r = requests.get(f'{API}/{path}.json', params={'limit': 100, 'offset': offset}, timeout=30)
            if r.status_code == 429:
                time.sleep(15 * (attempt + 1))
                continue
            r.raise_for_status()
            break
        data = r.json()['MRData']
        for race in data['RaceTable']['Races']:
            entry = races.setdefault(int(race['round']), {**race, 'rows': []})
            entry['rows'] += race.get('Results', race.get('QualifyingResults', []))
        offset += 100
        if offset >= int(data['total']):
            return races
        time.sleep(0.5)


def lap_seconds(s):
    if not s:
        return np.nan
    m, sec = s.split(':') if ':' in s else ('0', s)
    return int(m) * 60 + float(sec)


def is_wet_race(year, rnd):
    # Hava verisi F1 canlı zamanlamadan (Ergast limitine takılmaz); yoksa kuru varsay
    try:
        wd = f1api.weather_data(fastf1.get_session(year, rnd, 'R').api_path)
        rain = pd.Series(wd['Rainfall']).map(lambda v: str(v).lower() in ('true', '1'))
        return int(rain.mean() > 0.1)
    except Exception as e:
        weather_fails.append(f'{year} R{rnd}: {type(e).__name__}')
        return 0


now = pd.Timestamp.now()
frames, weather_fails = [], []
for year in YEARS:
    results = jolpica(f'{year}/results')
    quali = jolpica(f'{year}/qualifying')
    for rnd, race in sorted(results.items()):
        if pd.Timestamp(race['date']) >= now - pd.Timedelta(days=1):
            continue
        best = {q['Driver']['driverId']: pd.Series([lap_seconds(q.get(k)) for k in ('Q1', 'Q2', 'Q3')]).min()
                for q in quali.get(rnd, {'rows': []})['rows']}
        pole = np.nanmin(list(best.values())) if best else np.nan
        rows = pd.DataFrame([{
            'year': year,
            'round': rnd,
            'event': race['raceName'],
            'circuit': race['Circuit']['circuitId'],
            'date': pd.Timestamp(race['date']),
            'driver': x['Driver']['driverId'],
            'team': x['Constructor']['constructorId'],
            'grid': int(x['grid']),
            'finish': int(x['position']),
            'points': float(x['points']),
            'status': x['status'],
            'quali_gap_pct': (best.get(x['Driver']['driverId'], np.nan) / pole - 1) * 100,
        } for x in race['rows']])
        rows['is_wet'] = is_wet_race(year, rnd)
        frames.append(rows)
        print(f"✅ {year} R{rnd:02d} {race['raceName']}" + ('  🌧️' if rows['is_wet'].iat[0] else ''))

raw = pd.concat(frames, ignore_index=True)
raw.to_csv('f1_raw_results.csv', index=False)
print(raw.shape)
if weather_fails:
    print(f'⚠️ {len(weather_fails)} yarışta hava verisi alınamadı (kuru varsayıldı):', weather_fails[:5])

## 2. Özellik mühendisliği

Tüm özellikler **sadece o yarıştan önceki** verilerle hesaplanır (`shift(1)`), yani model tahmin anında bilinmeyen bir bilgiyi görmez. Aynı özellikler 2026'da canlı tahmin için de FastF1'den hesaplanabilir.

In [ ]:
# Takım isim değişikliklerini tek bir "soy" altında topla
TEAM_ALIASES = {
    'toro_rosso': 'rb', 'alphatauri': 'rb',
    'force_india': 'aston_martin', 'racing_point': 'aston_martin',
    'renault': 'alpine',
    'alfa': 'audi', 'sauber': 'audi',
}

df = raw.dropna(subset=['finish']).copy()
df['team'] = df['team'].replace(TEAM_ALIASES)
df['grid'] = df['grid'].fillna(20).replace(0, 20).clip(1, 22)          # pit lane start → arka sıra
df['dnf'] = (~df['status'].str.contains('Finished|Lap', case=False, na=False)).astype(int)
df['race_id'] = df['year'] * 100 + df['round']
df['gain'] = df['grid'] - df['finish']
df = df.sort_values(['race_id', 'finish']).reset_index(drop=True)

df['quali_gap_pct'] = df['quali_gap_pct'].fillna(df.groupby('race_id')['quali_gap_pct'].transform('max')).fillna(3.0)


def past_rolling(frame, by, col, window):
    return frame.groupby(by)[col].transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())

# --- Pilot formu ---
df['driver_form_5'] = past_rolling(df, 'driver', 'finish', 5)
df['driver_points_5'] = past_rolling(df, 'driver', 'points', 5)
df['driver_dnf_10'] = past_rolling(df, 'driver', 'dnf', 10)
df['driver_gain_5'] = past_rolling(df, 'driver', 'gain', 5)
df['driver_experience'] = df.groupby('driver').cumcount()
df['driver_track_avg'] = df.groupby(['driver', 'circuit'])['finish'].transform(lambda s: s.shift(1).expanding().mean())

# --- Takım formu (takım-yarış seviyesinde, aynı yarıştaki takım arkadaşı sızmasın) ---
team_race = (df.groupby(['race_id', 'team'])
               .agg(t_finish=('finish', 'mean'), t_points=('points', 'sum'), t_dnf=('dnf', 'mean'))
               .reset_index().sort_values('race_id'))
team_race['team_form_5'] = past_rolling(team_race, 'team', 't_finish', 5)
team_race['team_points_5'] = past_rolling(team_race, 'team', 't_points', 5)
team_race['team_dnf_10'] = past_rolling(team_race, 'team', 't_dnf', 10)
df = df.merge(team_race[['race_id', 'team', 'team_form_5', 'team_points_5', 'team_dnf_10']],
              on=['race_id', 'team'], how='left')

# --- Sezon içi şampiyona durumu (yarıştan önce) ---
df['season_points_before'] = df.groupby(['year', 'driver'])['points'].transform(lambda s: s.shift(1).cumsum()).fillna(0)
df['season_rank_before'] = df.groupby('race_id')['season_points_before'].rank(ascending=False, method='min')

# --- Pist karakteri: önceki yıllarda bu pistte ortalama pozisyon değişimi (sollama kolaylığı) ---
race_change = (df[df['dnf'] == 0].assign(abs_gain=lambda d: d['gain'].abs())
                 .groupby(['race_id', 'circuit'])['abs_gain'].mean().reset_index().sort_values('race_id'))
race_change['circuit_pos_change'] = race_change.groupby('circuit')['abs_gain'].transform(lambda s: s.shift(1).expanding().mean())
df = df.merge(race_change[['race_id', 'circuit_pos_change']], on='race_id', how='left')

# --- Eksik değerler (ilk yarışlar, yeni pilotlar) ---
df['driver_form_5'] = df['driver_form_5'].fillna(df['grid'])
df['driver_track_avg'] = df['driver_track_avg'].fillna(df['driver_form_5'])
df['team_form_5'] = df['team_form_5'].fillna(12)
for c in ['driver_points_5', 'team_points_5', 'driver_gain_5']:
    df[c] = df[c].fillna(0)
for c in ['driver_dnf_10', 'team_dnf_10']:
    df[c] = df[c].fillna(df['dnf'].mean())
df['circuit_pos_change'] = df['circuit_pos_change'].fillna(df['circuit_pos_change'].median())

FEATURES = [
    'grid', 'quali_gap_pct', 'is_wet', 'round',
    'driver_form_5', 'driver_points_5', 'driver_dnf_10', 'driver_gain_5',
    'driver_experience', 'driver_track_avg',
    'team_form_5', 'team_points_5', 'team_dnf_10',
    'season_points_before', 'season_rank_before', 'circuit_pos_change',
]
TARGET = 'finish'
print(df[FEATURES].describe().T[['mean', 'min', 'max']])

## 3. Zaman bazlı ayırma + baseline

In [ ]:
from scipy.stats import spearmanr

train = df[df['year'] <= 2024]
val = df[df['year'] == 2025]
test = df[df['year'] == 2026]
if test.empty:            # 2026 verisi yoksa 2025'i test olarak kullan
    train, val, test = df[df['year'] <= 2023], df[df['year'] == 2024], df[df['year'] == 2025]
print(f'train={len(train)}  val={len(val)}  test={len(test)}')


def evaluate(frame, pred):
    f = frame.assign(pred=np.asarray(pred))
    sp, win, pod = [], [], []
    for _, r in f.groupby('race_id'):
        sp.append(spearmanr(r['pred'], r['finish']).correlation)
        win.append(r.loc[r['pred'].idxmin(), 'finish'] == 1)
        pod.append(len(set(r.nsmallest(3, 'pred')['driver']) & set(r.nsmallest(3, 'finish')['driver'])) / 3)
    return {
        'MAE': round((f['pred'] - f['finish']).abs().mean(), 3),
        'Spearman': round(np.nanmean(sp), 3),
        'Kazanan isabet': round(np.mean(win), 3),
        'Podyum isabet': round(np.mean(pod), 3),
    }

results = {'Baseline (grid = bitiş)': evaluate(test, test['grid'])}
pd.DataFrame(results).T

## 4. Modeller: Random Forest, Gradient Boosting, XGBoost (GPU)

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb

Xtr, ytr = train[FEATURES], train[TARGET]
Xva, yva = val[FEATURES], val[TARGET]
Xte = test[FEATURES]

rf = RandomForestRegressor(n_estimators=400, max_depth=10, min_samples_leaf=5, n_jobs=-1, random_state=42)
gb = GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=42)
xg = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.03, max_depth=5, subsample=0.8,
                      colsample_bytree=0.8, tree_method='hist', device=DEVICE,
                      early_stopping_rounds=100, random_state=42)

rf.fit(Xtr, ytr)
gb.fit(Xtr, ytr)
xg.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
print('XGBoost best iteration:', xg.best_iteration)

WEIGHTS = {'rf': 0.40, 'gb': 0.35, 'xgb': 0.25}   # projedeki ensemble ağırlıklarıyla aynı mantık
preds = {'rf': rf.predict(Xte), 'gb': gb.predict(Xte), 'xgb': xg.predict(Xte)}
preds['ensemble'] = sum(preds[k] * w for k, w in WEIGHTS.items())

for name, p in preds.items():
    results[name] = evaluate(test, p)
pd.DataFrame(results).T

In [ ]:
import matplotlib.pyplot as plt

imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
imp.plot.barh(figsize=(7, 6), title='Random Forest - özellik önemi')
plt.tight_layout(); plt.show()

## 5. Tüm veriyle son eğitim ve kaydetme

In [ ]:
import joblib, sklearn
from datetime import datetime

X_all, y_all = df[FEATURES], df[TARGET]
rf_f = RandomForestRegressor(**rf.get_params()).fit(X_all, y_all)
gb_f = GradientBoostingRegressor(**gb.get_params()).fit(X_all, y_all)
xg_f = xgb.XGBRegressor(**{**xg.get_params(), 'n_estimators': xg.best_iteration + 1,
                           'early_stopping_rounds': None}).fit(X_all, y_all)

bundle = {
    'models': {'rf': rf_f, 'gb': gb_f},
    # XGBoost pickle'ı sürüm/GPU bağımlı → sürümden bağımsız JSON olarak sakla
    'xgb_json': bytes(xg_f.get_booster().save_raw('json')).decode(),
    'weights': WEIGHTS,
    'features': FEATURES,
    'team_aliases': TEAM_ALIASES,
    'test_metrics': results,
    'trained_on': f"{df['year'].min()}-{df['year'].max()} ({df['race_id'].nunique()} yarış)",
    'created': datetime.now().isoformat(timespec='seconds'),
    'versions': {'sklearn': sklearn.__version__, 'xgboost': xgb.__version__, 'fastf1': fastf1.__version__},
}
joblib.dump(bundle, 'f1_model.joblib', compress=3)
df.to_csv('f1_training_data.csv', index=False)
print('Kaydedildi. Lokal kurulumda aynı sürümleri kullan:')
print(f"  pip install scikit-learn=={sklearn.__version__} xgboost=={xgb.__version__}")

from google.colab import files
files.download('f1_model.joblib')
files.download('f1_training_data.csv')